In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
DATA_DIR = Path("data")
rt_path = DATA_DIR / "rtMatrix.txt"

print("File exists:", rt_path.exists())
print("File path:", rt_path)

File exists: True
File path: data/rtMatrix.txt


In [6]:
rt = pd.read_csv(rt_path, sep="\t", header=None)

print("Dataset shape:", rt.shape)
rt.head()

Dataset shape: (339, 5826)


,0,1,2,3,4,5,6,7,8,9,...,5816,5817,5818,5819,5820,5821,5822,5823,5824,5825
0,5.982,0.228,0.237,0.221,0.222,0.527,0.453,0.566,0.386,0.649,...,7.948,5.673,9.600,2.172,5.402,6.486,5.245,6.709,6.777,NaN
1,2.130,0.262,0.273,0.251,0.254,0.427,0.652,0.642,0.439,0.720,...,0.499,0.164,0.457,0.280,0.270,0.287,2.308,0.254,0.263,NaN
2,0.854,0.366,0.376,0.357,0.358,0.115,0.649,0.980,0.642,1.213,...,0.814,0.516,0.786,0.610,-1.000,-1.000,10.455,-1.000,-1.000,NaN
3,0.693,0.226,0.233,0.220,0.219,0.344,0.765,0.793,0.371,0.552,...,0.376,0.255,0.621,0.177,0.167,0.142,0.217,0.170,0.173,NaN
4,0.866,0.227,0.233,0.220,0.221,0.366,0.836,0.863,0.372,0.619,...,0.413,0.150,0.173,0.091,0.568,0.092,0.164,0.090,0.095,NaN


In [7]:
rt_clean = rt.replace(-1, np.nan)

print("Missing values:", rt_clean.isna().sum().sum())
print("Total values:", rt_clean.size)


Missing values: 101176
Total values: 1975014


In [8]:
max_value = rt_clean.max().max()

rt_clean = rt_clean / max_value

print("Maximum original value:", max_value)
print("Minimum normalised value:", rt_clean.min().min())
print("Maximum normalised value:", rt_clean.max().max())


Maximum original value: 19.999
Minimum normalised value: 5.0002500125006255e-05
Maximum normalised value: 1.0


In [9]:
total_values = rt_clean.size
missing_values = rt_clean.isna().sum().sum()

sparsity = missing_values / total_values

print("Total values:", total_values)
print("Missing values:", missing_values)
print(f"Sparsity: {sparsity:.4f}")
print(f"Sparsity percentage: {sparsity * 100:.2f}%")


Total values: 1975014
Missing values: 101176
Sparsity: 0.0512
Sparsity percentage: 5.12%


In [10]:
clean_path = DATA_DIR / "rt_clean.csv"

rt_clean.to_csv(clean_path, index=False)

print("Cleaned dataset saved to:", clean_path)

Cleaned dataset saved to: data/rt_clean.csv


In [11]:
np.random.seed(42)

rt_malicious = rt_clean.copy()

num_users = rt_malicious.shape[0]
num_malicious_users = int(np.ceil(num_users * 0.10))

malicious_user_indices = np.random.choice(
    rt_malicious.index,
    size=num_malicious_users,
    replace=False
)

print("Total users:", num_users)
print("Number of malicious users:", num_malicious_users)
print("Malicious user indices:")
print(malicious_user_indices)

Total users: 339
Number of malicious users: 34
Malicious user indices:
[284 116 113  42 126 231 181 278 336  25 124 221 261 155 109 172 118 325
 266   9 280  55 165 119 144  90 108 210 139 314 176  39 104 316]


In [12]:
for user_id in malicious_user_indices:
    rated_items = rt_malicious.loc[user_id].notna()
    
    attack_type = np.random.choice(["high", "low"])
    
    if attack_type == "high":
        rt_malicious.loc[user_id, rated_items] = 1.0
    else:
        rt_malicious.loc[user_id, rated_items] = 0.0

print("Malicious dataset created.")
rt_malicious.head()

Malicious dataset created.


,0,1,2,3,4,5,6,7,8,9,...,5816,5817,5818,5819,5820,5821,5822,5823,5824,5825
0,0.299115,0.011401,0.011851,0.011051,0.011101,0.026351,0.022651,0.028301,0.019301,0.032452,...,0.397420,0.283664,0.480024,0.108605,0.270114,0.324316,0.262263,0.335467,0.338867,NaN
1,0.106505,0.013101,0.013651,0.012551,0.012701,0.021351,0.032602,0.032102,0.021951,0.036002,...,0.024951,0.008200,0.022851,0.014001,0.013501,0.014351,0.115406,0.012701,0.013151,NaN
2,0.042702,0.018301,0.018801,0.017851,0.017901,0.005750,0.032452,0.049002,0.032102,0.060653,...,0.040702,0.025801,0.039302,0.030502,NaN,NaN,0.522776,NaN,NaN,NaN
3,0.034652,0.011301,0.011651,0.011001,0.010951,0.017201,0.038252,0.039652,0.018551,0.027601,...,0.018801,0.012751,0.031052,0.008850,0.008350,0.007100,0.010851,0.008500,0.008650,NaN
4,0.043302,0.011351,0.011651,0.011001,0.011051,0.018301,0.041802,0.043152,0.018601,0.030952,...,0.020651,0.007500,0.008650,0.004550,0.028401,0.004600,0.008200,0.004500,0.004750,NaN


In [13]:
malicious_path = DATA_DIR / "rt_malicious.csv"

rt_malicious.to_csv(malicious_path, index=False)

print("Malicious dataset saved to:", malicious_path)

Malicious dataset saved to: data/rt_malicious.csv


In [14]:
labels = pd.DataFrame({
    "user_id": malicious_user_indices,
    "is_malicious": 1
})

labels_path = DATA_DIR / "malicious_users_10_percent.csv"
labels.to_csv(labels_path, index=False)

print("Malicious user labels saved to:", labels_path)

Malicious user labels saved to: data/malicious_users_10_percent.csv


In [15]:
import os

for file in DATA_DIR.iterdir():
    print(file)

data/rt_clean.csv
data/malicious_users_10_percent.csv
data/rtMatrix.txt
data/processed
data/rt_malicious.csv
data/raw


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

rt_path = DATA_DIR / "rtMatrix.txt"

# Load original response time matrix
rt = pd.read_csv(rt_path, sep="\t", header=None)

# Replace missing values
rt = rt.replace(-1, np.nan)

# Normalise response time
max_rt = rt.max().max()
rt_normalised = rt / max_rt

# Convert response time into quality score
# Lower response time = better quality
rt_clean = 1 - rt_normalised

# Keep missing values as NaN
rt_clean = rt_clean.where(rt.notna(), np.nan)

# Save corrected clean file
rt_clean.to_csv(DATA_DIR / "rt_clean.csv", index=False)

print("Corrected rt_clean.csv saved.")
print("Shape:", rt_clean.shape)
print("Minimum value:", rt_clean.min().min())
print("Maximum value:", rt_clean.max().max())
print("Missing values:", rt_clean.isna().sum().sum())

Corrected rt_clean.csv saved.
Shape: (339, 5826)
Minimum value: 0.0
Maximum value: 0.999949997499875
Missing values: 101176


In [2]:
np.random.seed(42)

rt_malicious = rt_clean.copy()

num_users = rt_malicious.shape[0]
num_malicious_users = int(np.ceil(num_users * 0.10))

malicious_user_indices = np.random.choice(
    rt_malicious.index,
    size=num_malicious_users,
    replace=False
)

for user_id in malicious_user_indices:
    rated_items = rt_malicious.loc[user_id].notna()
    attack_type = np.random.choice(["high", "low"])
    
    if attack_type == "high":
        rt_malicious.loc[user_id, rated_items] = 1.0
    else:
        rt_malicious.loc[user_id, rated_items] = 0.0

rt_malicious.to_csv(DATA_DIR / "rt_malicious.csv", index=False)

labels = pd.DataFrame({
    "user_id": malicious_user_indices,
    "is_malicious": 1
})

labels.to_csv(DATA_DIR / "malicious_users_10_percent.csv", index=False)

print("Corrected malicious dataset saved.")
print("Total users:", num_users)
print("Malicious users:", num_malicious_users)

Corrected malicious dataset saved.
Total users: 339
Malicious users: 34
